# Structural analysis of spherical diagrams

This notebook starts with the saved spherical representatives, visualizes all representatives for `N=4`, and tests the main structural patterns across `N=1..5`.

In [ ]:
from __future__ import annotations

import csv
import pickle
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd().resolve()
while not (project_root / 'src').exists() and project_root != project_root.parent:
    project_root = project_root.parent
if not (project_root / 'src').exists():
    raise FileNotFoundError('Could not find the src folder.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.coxeter import (
    LABELS,
    NO_EDGE,
    canonical_code_and_automorphisms,
    connected_components,
    edge_list,
    is_acyclic,
    num_edges,
    orbit_size,
    spherical_margin,
)

data_dir = project_root / 'data'
results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)

representatives_by_n = {}
for path in sorted(data_dir.glob('representatives_n*.pkl')):
    n = int(path.stem.split('n')[-1])
    with path.open('rb') as handle:
        representatives_by_n[n] = pickle.load(handle)

if not representatives_by_n:
    raise FileNotFoundError('No saved representatives were found.')

print(f'Loaded representatives for N={sorted(representatives_by_n)}')
print(f'Project root: {project_root}')

## Gallery of the `N=4` representatives

The edge labels are Coxeter labels: `3` is shown explicitly here, while `2` means that no edge is present.

In [ ]:
def draw_diagram(ax, matrix, title):
    n = matrix.shape[0]
    if n == 1:
        positions = {0: np.array([0.0, 0.0])}
    elif n == 2:
        positions = {0: np.array([-0.55, 0.0]), 1: np.array([0.55, 0.0])}
    else:
        angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, n, endpoint=False)
        positions = {i: np.array([np.cos(angle), np.sin(angle)]) for i, angle in enumerate(angles)}

    for i, j in edge_list(matrix):
        start, end = positions[i], positions[j]
        ax.plot([start[0], end[0]], [start[1], end[1]], color='#263238', linewidth=1.5, zorder=1)
        midpoint = (start + end) / 2
        label = str(int(matrix[i, j]))
        ax.text(midpoint[0], midpoint[1], label, fontsize=7, ha='center', va='center',
                bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 1.0}, zorder=3)

    coords = np.array([positions[i] for i in range(n)])
    ax.scatter(coords[:, 0], coords[:, 1], s=90, color='#1565c0', edgecolor='white', linewidth=0.8, zorder=2)
    for i, (x, y) in positions.items():
        ax.text(x, y, str(i), color='white', fontsize=8, ha='center', va='center', zorder=4)
    ax.set_title(title, fontsize=8)
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.25, 1.25)
    ax.set_aspect('equal')
    ax.axis('off')

n4_representatives = representatives_by_n.get(4, [])
page_size = 24
for page_start in range(0, len(n4_representatives), page_size):
    page = n4_representatives[page_start:page_start + page_size]
    columns = 6
    rows = int(np.ceil(len(page) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(13, 2.2 * rows), squeeze=False)
    for local_index, matrix in enumerate(page):
        ax = axes.flat[local_index]
        absolute_index = page_start + local_index
        draw_diagram(ax, matrix, f'#{absolute_index} | edges={num_edges(matrix)}')
    for ax in axes.flat[len(page):]:
        ax.axis('off')
    fig.suptitle(f'Spherical representatives for N=4 (page {page_start // page_size + 1})', y=1.01)
    fig.tight_layout()
    output_path = results_dir / f'spherical_gallery_n4_page_{page_start // page_size + 1}.png'
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    print(f'Saved {output_path}')
    plt.show()

## Structural features

The tests below use only isomorphism-invariant quantities, so a result for a representative applies to every labelled diagram in its orbit.

In [ ]:
feature_rows = []
for n, representatives in sorted(representatives_by_n.items()):
    for representative_index, matrix in enumerate(representatives):
        _, automorphism_count = canonical_code_and_automorphisms(matrix)
        labels = [int(matrix[i, j]) for i, j in edge_list(matrix)]
        degrees = [sum(matrix[i, j] != NO_EDGE for j in range(n)) for i in range(n)]
        feature_rows.append({
            'n': n,
            'representative_index': representative_index,
            'n_edges': num_edges(matrix),
            'n_components': connected_components(matrix),
            'acyclic': is_acyclic(matrix),
            'degree_sequence': '-'.join(map(str, sorted(degrees, reverse=True))),
            'edge_labels': '-'.join(map(str, sorted(labels))) if labels else 'none',
            'spherical_margin': spherical_margin(matrix),
            'automorphism_count': automorphism_count,
            'orbit_size': orbit_size(matrix),
        })

feature_fields = list(feature_rows[0])
feature_path = results_dir / 'spherical_representative_features.csv'
with feature_path.open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=feature_fields)
    writer.writeheader()
    writer.writerows(feature_rows)
print(f'Saved representative features to {feature_path}')

In [ ]:
summary_rows = []
for n in sorted(representatives_by_n):
    rows = [row for row in feature_rows if row['n'] == n]
    component_counts = Counter(row['n_components'] for row in rows)
    summary_rows.append({
        'n': n,
        'n_representatives': len(rows),
        'n_acyclic': sum(row['acyclic'] for row in rows),
        'n_connected': sum(row['n_components'] == 1 for row in rows),
        'min_edges': min(row['n_edges'] for row in rows),
        'max_edges': max(row['n_edges'] for row in rows),
        'component_counts': '; '.join(f'{key}:{value}' for key, value in sorted(component_counts.items())),
        'min_spherical_margin': min(row['spherical_margin'] for row in rows),
        'max_spherical_margin': max(row['spherical_margin'] for row in rows),
    })

summary_path = results_dir / 'spherical_structure_summary.csv'
with summary_path.open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)
print(f'Saved structural summary to {summary_path}')
for row in summary_rows:
    print(row)

## Main structural checks

For a forest with `c` connected components on `N` vertices, the number of edges is `N - c`.

In [ ]:
assert all(row['acyclic'] for row in feature_rows), 'A spherical representative with a cycle was found.'
assert all(row['n_edges'] == row['n'] - row['n_components'] for row in feature_rows), (
    'The forest identity edges = N - components failed.'
)

benchmark_path = results_dir / 'exhaustive_summary.csv'
if benchmark_path.exists():
    benchmark_rows = {}
    with benchmark_path.open(newline='') as handle:
        for row in csv.DictReader(handle):
            benchmark_rows[int(row['n'])] = int(row['n_spherical_classes'])
    for n, representatives in representatives_by_n.items():
        assert len(representatives) == benchmark_rows[n], f'Class count mismatch at N={n}'

print('All spherical representatives are forests.')
print('The representative counts agree with the exhaustive benchmark.')

## Summary plots

In [ ]:
n_values = [row['n'] for row in summary_rows]
min_edges = [row['min_edges'] for row in summary_rows]
max_edges = [row['max_edges'] for row in summary_rows]
min_margin = [row['min_spherical_margin'] for row in summary_rows]
max_margin = [row['max_spherical_margin'] for row in summary_rows]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].fill_between(n_values, min_edges, max_edges, color='#90caf9', alpha=0.8, label='representative range')
axes[0].plot(n_values, min_edges, 'o-', color='#1565c0')
axes[0].plot(n_values, max_edges, 'o-', color='#0d47a1')
axes[0].set_title('Number of edges by N')
axes[0].set_xlabel('N')
axes[0].set_ylabel('Edges')
axes[0].grid(alpha=0.3)

axes[1].fill_between(n_values, min_margin, max_margin, color='#ffcc80', alpha=0.8)
axes[1].plot(n_values, min_margin, 'o-', color='#e65100')
axes[1].plot(n_values, max_margin, 'o-', color='#bf360c')
axes[1].set_title('Spherical margin by N')
axes[1].set_xlabel('N')
axes[1].set_ylabel('Smallest Cartan eigenvalue')
axes[1].grid(alpha=0.3)

fig.tight_layout()
plot_path = results_dir / 'spherical_structure_summary.png'
fig.savefig(plot_path, dpi=200)
print(f'Saved summary plot to {plot_path}')
plt.show()